Update image with cellular automaton
- find rules

## Import modules

In [2]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path
from datetime import date
# from dataclasses import dataclass, field
# import itertools


# import 3rd-party modules
import cv2
import numpy as np
# from numba import njit
# import ray
# from pygifsicle import optimize


# import local modules
from utils.renderer.giffer import create_gif
from utils.renderer.videographer import create_video
from utils.project_manager import Project
from utils.renderer.resizer import get_interpolation
from utils.renderer.resizer import resize_with_pad, resize_with_crop

## Set up project

In [55]:
# create project
project = Project(project_dir="assets/images/mosaic/cellular_automaton")

## Define functions

In [58]:
def find_nearest_multiple(x, base):
    """
    Function to find the nearest multiple of base given an integer
    """
    return base * round(x/base)

In [82]:
def find_surrounding_cells(cell_yx, img_shape):
    """
    Function to find yx coords of surrounding cells in image.
    """
    
    # create empty list to store yx coords of surrounding cells
    surrounding_yxs = []

    # unpack cell yx coords
    y, x = cell_yx

    # set possible surrounding indexes
    surrounding_idxs = np.array([
    [y-1, x-1],
    [y-1, x],
    [y-1, x+1],
    [y, x-1],
    [y, x+1],
    [y+1, x-1],
    [y+1, x],
    [y+1, x+1],
    ])

    # get only surrounding indexes inside image
    for surrounding_idx in surrounding_idxs:
        s_y_idx, s_x_idx = surrounding_idx
        if not ((surrounding_idx < 0).any() or s_y_idx >= img_shape[0] or s_x_idx >= img_shape[1]):
            surrounding_yxs.append(surrounding_idx)
            
    return surrounding_yxs

def mean_automaton_rules(cell_hue, surrounding_cells_hues):
    """
    Function to define cellular automaton rules to update the current cell
    based on its state and states from surrounding cells
    """
    
    surrounding_cells_hues_mean = np.mean(surrounding_cells_hues)

    if cell_hue < surrounding_cells_hues_mean:
        return surrounding_cells_hues_mean

In [96]:
def run_cellular_automaton(
    dest_img, automaton_rules_fct, nb_rows, nb_cols, col_range=None,
    ):
    """
    Function to run a cellular_automaton on an image
    """

    nb_updates = 0

    # get img height & width
    dest_img_height, dest_img_width = dest_img.shape[:2]

    # get best number of rows and cols to cover the whole img
    out_img_height = find_nearest_multiple(dest_img_height, nb_rows)
    out_img_width = find_nearest_multiple(dest_img_width, nb_cols)

    # resize image to recreate so that the grid can cover the whole image
    # get interpolation
    interpolation = get_interpolation(src_img_shape=(dest_img_height, dest_img_width), out_img_shape=(out_img_height, out_img_width))
    dest_img = cv2.resize(dest_img, (out_img_width, out_img_height), interpolation=interpolation)

    # get cell height & width
    cell_height, cell_width = out_img_height//nb_rows, out_img_width//nb_cols

    # get list of grid positions
    if col_range is not None:
        grid_positions = [(grid_y, grid_x) for grid_x in range(*col_range) for grid_y in range(nb_rows)]
        # out_img = np.zeros_like(dest_img[:,col_range[0]:col_range[1]])
    else:
        grid_positions = [(grid_y, grid_x) for grid_x in range(nb_cols) for grid_y in range(nb_rows)]
        # out_img = np.zeros_like(dest_img)

    # get index range of grid positions
    grid_positions_idxs = np.arange(len(grid_positions))
    # # choose random seed to recreate same shuffle or change it to see if you get better results
    # np.random.seed(1111)
    # shuffle grid positions (otherwise the first grids from top will get the best matching images)
    # np.random.shuffle(grid_positions_idxs)

    # convert bgr image to hsv
    dest_img = cv2.cvtColor(dest_img, cv2.COLOR_BGR2HSV)

    # iterate over each grid position
    for i in grid_positions_idxs:

        grid_y, grid_x = grid_positions[i]
        
        # get hues of region of interest in img to recreate
        y = grid_y * cell_height
        x = grid_x * cell_width
        dest_roi_hues = dest_img[y:y+cell_height, x:x+cell_width, 0]
        
        # get mean of roi
        dest_roi_hues_mean = np.mean(dest_roi_hues)

        # find surrounding cells
        surrounding_cells_grid_yxs = find_surrounding_cells((grid_y, grid_x), (nb_rows, nb_cols))

        # create empty list to store hue values of surrounding cells
        surrounding_cells_hues = []

        # get surrounding cells rois
        for surrounding_cells_grid_yx in surrounding_cells_grid_yxs:

            s_grid_y, s_grid_x = surrounding_cells_grid_yx

            # get hues of surrounding regions of interest in img to recreate
            s_y = s_grid_y * cell_height
            s_x = s_grid_x * cell_width
            dest_s_roi_hues = dest_img[s_y:s_y+cell_height, s_x:s_x+cell_width, 0]
        
            # get mean of surrounding roi
            dest_s_roi_hues_mean = np.mean(dest_s_roi_hues)

            # append to list of hue values of surrounding cells
            surrounding_cells_hues.append(dest_s_roi_hues_mean)

        # apply automaton_rules_fct
        dest_roi_hues_update = automaton_rules_fct(dest_roi_hues_mean, surrounding_cells_hues)

        # if cell updates, apply new hue values
        if dest_roi_hues_update:
            dest_img[y:y+cell_height, x:x+cell_width, 0] = int(dest_roi_hues_update)

            # record that there has been an update
            nb_updates += 1

    # convert bgr image to hsv
    dest_img = cv2.cvtColor(dest_img, cv2.COLOR_HSV2BGR)

    return nb_updates, dest_img

In [90]:
# set dest image path (i.e path of image to recreate)
dest_img_path = Path("/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/images/pixel_transfers/db/edvard_munch-the-scream.jpeg")

# get dest image
dest_img = cv2.imread(str(dest_img_path))

In [95]:
NB_ROWS = 300
NB_COLS = 300

# set output image directory
out_img_dir = f"random_{dest_img_path.stem}__{NB_ROWS*NB_COLS}"

# make output image directory
project.make_dir(out_img_dir)

In [97]:
# get current date
today = date.today().strftime("%Y%m%d")

NB_IMGS = 200
nb_updates_threshold = 10

out_img = dest_img.copy()

for i in range(0, NB_IMGS):

    # recreate image with mosaic of shape
    nb_updates, out_img = run_cellular_automaton(out_img, mean_automaton_rules, NB_ROWS, NB_COLS)

    # stop writing images if no more updates
    if nb_updates < nb_updates_threshold:
        break

    # set output image path
    out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"{NB_ROWS*NB_COLS}_{today}_{i:09d}.jpg")

    # save output image & metadata
    cv2.imwrite(out_img_path, out_img)

KeyboardInterrupt: 

In [99]:
# get img height & width
dest_img_height, dest_img_width = dest_img.shape[:2]

# set gif path
gif_path = project.project_dir / f"{out_img_dir}.gif"

# create gif
create_gif(img_dir=project.out_img_dir_dict[out_img_dir], out_path=gif_path, out_img_shape=(int(dest_img_height//2),int(dest_img_width//2)),
sort_img_list=True, duplicate_start_img_amount=0, duplicate_end_img_amount=0)

In [98]:
nb_updates

66614